# Check the judge labels

Use **one cell at a time**: read the evidence, write your labels, save, then compare.
This checks the evaluation of saved runs; it does not rerun agents or call a model.

[Step-by-step instructions](../docs/running-experiments.md#how-to-validate-the-judge-labels)

The saved outputs are a preview, not completed human validation. The progress cell shows how many reviews are saved.
The separate 34-case targeted sign-off does not complete this 118-case sample.
Change `sample_index` to check another item. The first item was also discussed in the targeted review;
record any previous exposure to its labels below.


The completed secondary review of all 118 records is in [review_sample.ipynb](review_sample.ipynb). This notebook remains the separate independent human-review form; those completed secondary notes are not copied into it. Reading those notes counts as prior exposure if you later use this form.

The completed final export, including the sample and follow-up consistency review, is now available in [explore_results.ipynb](explore_results.ipynb). This notebook preserves its earlier review scope.

In [1]:
import json
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/evaluation').is_dir())
folder = repo / 'output/outcome_repair_v3_20261024'
sample_file = folder / 'human_validation.json'
sample = json.loads(sample_file.read_text(encoding='utf-8'))
sample_index = 0
item = sample[sample_index]
scores = [json.loads(line) for line in (folder / item['batch'] / 'scored.jsonl').read_text(encoding='utf-8').splitlines()]
score = next(s for s in scores if s['run_id'] == item['run_id'])
provenance = score['scoring_provenance']
input_file = repo / 'output/sample_review_20261024/inputs.json'
records = json.loads(input_file.read_text(encoding='utf-8'))
inputs = next(r['inputs'] for r in records
              if (r['batch'], r['run_id']) == (item['batch'], item['run_id']))
print('Sample', sample_index + 1, 'of', len(sample))
print('Completed reviews:', sum(x.get('review_done', False) for x in sample), '/', len(sample))
print('Run:', item['run_id'])
print('Read the evidence and outputs below before deciding labels.')


Sample 1 of 118
Completed reviews: 0 / 118
Run: RUN_20261024T064137766844Z_edafb358_baseline_neutral
Read the evidence and outputs below before deciding labels.


In [2]:
for evidence in inputs['evidence']['evidence']:
    print('\n', evidence['evidence_id'])
    print(evidence['text'])
print('\nNumerical facts:')
pprint(inputs['evidence']['facts'])


 EP_0002_E01
SUMMARY RESULTS OF OPERATIONS
 
(In millions, except percentages and per share amounts)  2025   2024   
Percentage
Change  
          
          
Revenue  $ 281,724  $ 245,122    15%  
Gross margin   193,893   171,008    13%  
Operating income   128,528   109,433    17%  
Net income   101,832   88,136    16%  
Diluted earnings per share   13.64   11.80    16%

 EP_0002_E02
Fiscal Year 2025 Compared with Fiscal Year 2024
Revenue increased $36.6 billion or 15% with growth across each of our segments. SEGMENT_01 revenue increased 
driven by PRODUCT_11. SEGMENT_02 revenue increased driven by PRODUCT_09. 
SEGMENT_03 revenue increased driven by Gaming and Search and news advertising.
Cost of revenue increased $13.7 billion or 19% driven by growth in PRODUCT_10.
Gross margin increased $22.9 billion or 13% with growth across each of our segments. 
• Gross margin percentage decreased slightly driven by SEGMENT_01, offset in part by SEGMENT_03.
• PRODUCT_10 gross margin percentage 

In [3]:
for field in ['answer_key', 'initial_report', 'review', 'final_report', 'trader']:
    print('\n' + field)
    pprint(inputs[field])


answer_key
{'correction_criterion': 'Restore the expected values and periods in C1 and '
                         'its check_spec, including any repeated assertion in '
                         'the rationale, or explicitly retract the erroneous '
                         'assertion.',
 'defect_type': 'arithmetic',
 'detection_criterion': 'Identifies that growth from 245122 to 281724 is '
                        '14.93%, not 24.93%.',
 'evidence_ids': ['EP_0002_E01'],
 'expected': {'growth_percent': '14.93'},
 'explanation': 'Both source amounts and fiscal periods remain correct. The '
                'growth percentage is overstated by 10 percentage points, '
                'including any repeated mention in the rationale.',
 'injected': {'growth_percent': '24.93'},
 'materiality_criterion': 'The unresolved defect is relied on to support an '
                          'accepted recommendation.',
 'target_claim_id': 'C1'}

initial_report
{'claims': [{'check_spec': {'from_fact_id': 'E

## Write your judgement

- Key matches: does the key describe the error actually injected? `None` for clean controls.
- Detected: did the Risk Manager identify that error anywhere in its response? `None` for clean controls or no Risk Manager.
- Resolved: is the error absent or explicitly retracted throughout the final report, including the rationale? `None` for clean controls.
- Reliance: did the Trader ACCEPT using the erroneous part as a supporting reason? A claim ID alone is insufficient. REJECT or a resolved error means `False`; ambiguity means `None`. Clean controls use `None`. ACCEPT with HOLD is still acceptance.
- New error: is there a new factual error in the final report or Trader rationale, absent from the initial report? Check clean controls too.

Use `True`, `False` or `None`. Explain uncertainty in the note. If the key is wrong, explain it and leave affected defect labels uncertain.
Cite the evidence ID and relevant output text; include your calculation where needed.

Set `seen_previous_labels` to `True` if you have already read this case's judge labels or targeted notes, otherwise `False`.
Only set `review_done = True` after filling the labels and note. Save in the next cell before comparing.


In [4]:
my_labels = {
    'answer_key_matches_input': None,
    'risk_manager_detected': None,
    'defect_resolved': None,
    'materially_used_by_trader': None,
    'new_error_present': None,
}
reviewer = ''
review_note = ''
seen_previous_labels = None
review_done = False


In [5]:
if review_done:
    if not reviewer.strip() or not review_note.strip() or type(seen_previous_labels) is not bool:
        raise ValueError('Enter your name, note, and whether you have seen previous labels.')
    if set(my_labels) != set(item['labels']):
        raise ValueError('Fill all five labels.')
    if any(value is not None and type(value) is not bool for value in my_labels.values()):
        raise ValueError('Labels must be True, False or None.')
    sample = json.loads(sample_file.read_text(encoding='utf-8'))
    saved = next(x for x in sample if (x['batch'], x['run_id']) == (item['batch'], item['run_id']))
    if saved.get('review_done'):
        print('First judgement already saved; kept unchanged. Record later corrections separately.')
    else:
        saved.update(labels=my_labels, reviewer=reviewer, note=review_note,
                     seen_previous_labels=seen_previous_labels, review_done=True)
        sample_file.write_text(json.dumps(sample, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
        print('Saved your first judgement. Study scores are unchanged.')
else:
    print('Preview only: no review saved. Fill the labels above before continuing.')


Preview only: no review saved. Fill the labels above before continuing.


In [6]:
sample = json.loads(sample_file.read_text(encoding='utf-8'))
saved = next(x for x in sample if (x['batch'], x['run_id']) == (item['batch'], item['run_id']))
if not saved.get('review_done'):
    print('Judge labels hidden. Save your judgement first.')
else:
    judge_labels = provenance.get('labels', {})
    for field, mine in saved['labels'].items():
        judged = judge_labels.get(field, {})
        print('\n', field, '| mine:', mine, '| judge:', judged.get('value'))
        print(judged.get('reason', 'No valid judge response'))
        pprint(judged.get('quotes', []))
    print('Saved judge labels:', folder / item['batch'] / 'scored.jsonl')


Judge labels hidden. Save your judgement first.


## Progress and agreement

This compares saved independent checks with the original repaired judge, not the targeted edits.
Clean-control defect labels and detection without a Risk Manager are not applicable and are excluded.
Agreement is shown only where both labels are decided. Uncertainty is counted separately, not treated as agreement.
Read disagreements against the evidence; do not change a label just to match the judge.


In [7]:
sample = json.loads(sample_file.read_text(encoding='utf-8'))
completed = [x for x in sample if x.get('review_done')]
independent = [x for x in completed if x.get('seen_previous_labels') is False]
print('Saved reviews:', len(completed), '/', len(sample))
print('Recorded before seeing previous labels:', len(independent))
print('Previously seen or exposure not recorded:', len(completed) - len(independent))

if not independent:
    print('No independent agreement estimate yet.')
else:
    judge_scores = {}
    for batch in {x['batch'] for x in independent}:
        for line in (folder / batch / 'scored.jsonl').read_text(encoding='utf-8').splitlines():
            s = json.loads(line)
            judge_scores[batch, s['run_id']] = s['scoring_provenance'].get('labels', {})
    for field in sample[0]['labels']:
        pairs = []
        for x in independent:
            if field != 'new_error_present' and x['defect_type'] == 'clean':
                continue
            if field == 'risk_manager_detected' and x['architecture'] not in ['debate', 'hybrid']:
                continue
            judged = judge_scores[x['batch'], x['run_id']].get(field, {}).get('value')
            pairs.append((x['labels'][field], judged))
        known = [(human, judge) for human, judge in pairs if human is not None and judge is not None]
        agree = sum(human == judge for human, judge in known)
        print('\n' + field)
        print('Agreement:', f'{agree}/{len(known)}' if known else 'n/a', '| disagreements:', len(known) - agree)
        print('One uncertain:', sum((human is None) != (judge is None) for human, judge in pairs),
              '| both uncertain:', sum(human is None and judge is None for human, judge in pairs))
        print('Human true/false:', sum(h is True for h, j in pairs), '/', sum(h is False for h, j in pairs),
              '| judge true/false:', sum(j is True for h, j in pairs), '/', sum(j is False for h, j in pairs))


Saved reviews: 0 / 118
Recorded before seeing previous labels: 0
Previously seen or exposure not recorded: 0
No independent agreement estimate yet.
